# 01. 신경망 기초 (Neural Networks)

## 학습 목표
- Perceptron에서 MLP로의 발전 과정 이해
- 활성화 함수의 역할과 종류 파악
- 2-layer NN을 NumPy로 직접 구현 (forward + backward)
- 같은 모델을 PyTorch nn.Module로 구현하여 비교

## 참고 자료
- [3Blue1Brown - Neural Networks](https://www.youtube.com/playlist?list=PLZHQObOWTQDNU6R1_67000Dx_ZCJB-3pi)
- [Michael Nielsen - Neural Networks and Deep Learning](http://neuralnetworksanddeeplearning.com/)

---

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

## 1. Perceptron

퍼셉트론은 가장 단순한 신경망 단위. 입력에 가중치를 곱하고 합산한 뒤, 임계값을 넘으면 1, 아니면 0을 출력한다.

$$y = \begin{cases} 1 & \text{if } \sum_i w_i x_i + b \geq 0 \\ 0 & \text{otherwise} \end{cases}$$

**ML에서의 의미**: 퍼셉트론은 선형 분류기. 입력 공간에 직선(초평면)을 그어 두 클래스를 분리한다.

In [ ]:
class Perceptron:
    """단일 퍼셉트론 구현"""
    def __init__(self, weights, bias):
        self.weights = np.array(weights, dtype=float)
        self.bias = bias
    
    def forward(self, x):
        z = np.dot(self.weights, x) + self.bias
        return 1 if z >= 0 else 0

# AND 게이트: 둘 다 1일 때만 1
and_gate = Perceptron(weights=[1, 1], bias=-1.5)
print("=== AND Gate ===")
for x1, x2 in [(0,0), (0,1), (1,0), (1,1)]:
    print(f"  ({x1}, {x2}) -> {and_gate.forward([x1, x2])}")

# OR 게이트: 하나라도 1이면 1
or_gate = Perceptron(weights=[1, 1], bias=-0.5)
print("\n=== OR Gate ===")
for x1, x2 in [(0,0), (0,1), (1,0), (1,1)]:
    print(f"  ({x1}, {x2}) -> {or_gate.forward([x1, x2])}")

# NAND 게이트: AND의 반대
nand_gate = Perceptron(weights=[-1, -1], bias=1.5)
print("\n=== NAND Gate ===")
for x1, x2 in [(0,0), (0,1), (1,0), (1,1)]:
    print(f"  ({x1}, {x2}) -> {nand_gate.forward([x1, x2])}")

### XOR 문제: 퍼셉트론의 한계

XOR 게이트는 두 입력이 다를 때 1을 출력한다. 하지만 단일 퍼셉트론(=직선 하나)으로는 XOR을 분류할 수 없다.

| x1 | x2 | XOR |
|----|----|-----|
| 0  | 0  | 0   |
| 0  | 1  | 1   |
| 1  | 0  | 1   |
| 1  | 1  | 0   |

직선 하나로 (0,1), (1,0)을 (0,0), (1,1)과 분리할 수 없다 -- **선형 분리 불가능** 문제.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# AND: 선형 분리 가능
ax = axes[0]
ax.scatter([0, 0, 1], [0, 1, 0], c='red', s=100, label='0', zorder=5)
ax.scatter([1], [1], c='blue', s=100, label='1', zorder=5)
x_line = np.linspace(-0.5, 1.5, 100)
ax.plot(x_line, 1.5 - x_line, 'k--', alpha=0.5, label='decision boundary')
ax.set_xlim(-0.5, 1.5); ax.set_ylim(-0.5, 1.5)
ax.set_aspect('equal'); ax.grid(True, alpha=0.3)
ax.set_title('AND (linearly separable)'); ax.legend(fontsize=8)

# OR: 선형 분리 가능
ax = axes[1]
ax.scatter([0], [0], c='red', s=100, label='0', zorder=5)
ax.scatter([0, 1, 1], [1, 0, 1], c='blue', s=100, label='1', zorder=5)
ax.plot(x_line, 0.5 - x_line, 'k--', alpha=0.5, label='decision boundary')
ax.set_xlim(-0.5, 1.5); ax.set_ylim(-0.5, 1.5)
ax.set_aspect('equal'); ax.grid(True, alpha=0.3)
ax.set_title('OR (linearly separable)'); ax.legend(fontsize=8)

# XOR: 선형 분리 불가능
ax = axes[2]
ax.scatter([0, 1], [0, 1], c='red', s=100, label='0', zorder=5)
ax.scatter([0, 1], [1, 0], c='blue', s=100, label='1', zorder=5)
ax.set_xlim(-0.5, 1.5); ax.set_ylim(-0.5, 1.5)
ax.set_aspect('equal'); ax.grid(True, alpha=0.3)
ax.set_title('XOR (NOT linearly separable)'); ax.legend(fontsize=8)
ax.text(0.5, -0.3, 'No single line can separate!', ha='center', fontsize=9, color='red')

plt.tight_layout()
plt.show()

---
## 2. Multi-Layer Perceptron (MLP)

XOR 문제를 해결하려면 **hidden layer**가 필요하다. 여러 퍼셉트론을 층으로 쌓으면 비선형 결정 경계를 만들 수 있다.

**핵심 아이디어**: XOR = NAND AND (OR)을 조합하면 만들 수 있다.

$$\text{XOR}(x_1, x_2) = \text{AND}(\text{NAND}(x_1, x_2),\; \text{OR}(x_1, x_2))$$

In [ ]:
# XOR = AND(NAND(x1, x2), OR(x1, x2))
print("=== XOR via MLP (NAND + OR -> AND) ===")
print(f"{'x1':>3} {'x2':>3} | {'NAND':>5} {'OR':>3} | {'XOR':>4}")
print("-" * 30)
for x1, x2 in [(0,0), (0,1), (1,0), (1,1)]:
    # Hidden layer
    h1 = nand_gate.forward([x1, x2])  # NAND
    h2 = or_gate.forward([x1, x2])    # OR
    # Output layer
    out = and_gate.forward([h1, h2])   # AND
    print(f"{x1:>3} {x2:>3} | {h1:>5} {h2:>3} | {out:>4}")

### Hidden Layer의 역할

Hidden layer는 입력 공간을 **비선형 변환**하여 원래 선형 분리 불가능했던 문제를 분리 가능하게 만든다.

아래에서 XOR 데이터가 hidden layer를 통과하면 어떻게 변환되는지 시각화한다.

In [ ]:
# 연속적인 활성화 함수(sigmoid)로 XOR MLP 구현
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

# 학습된 가중치 (미리 설정)
W1 = np.array([[20, 20], [20, 20]])  # hidden layer weights
b1 = np.array([-30, -10])             # hidden layer bias
W2 = np.array([-20, 20])              # output weights
b2 = np.array([-10])                  # output bias

# XOR 데이터
X_xor = np.array([[0,0], [0,1], [1,0], [1,1]])
y_xor = np.array([0, 1, 1, 0])

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# 왼쪽: 원래 입력 공간
ax = axes[0]
colors = ['red' if y == 0 else 'blue' for y in y_xor]
ax.scatter(X_xor[:, 0], X_xor[:, 1], c=colors, s=200, zorder=5, edgecolors='black')
for i, (x, y) in enumerate(X_xor):
    ax.annotate(f'XOR={y_xor[i]}', (x, y), textcoords='offset points',
                xytext=(10, 10), fontsize=10)
ax.set_xlabel('x1'); ax.set_ylabel('x2')
ax.set_title('Input Space (linearly inseparable)')
ax.set_xlim(-0.5, 1.5); ax.set_ylim(-0.5, 1.5)
ax.grid(True, alpha=0.3); ax.set_aspect('equal')

# 오른쪽: hidden layer 출력 공간
ax = axes[1]
H = sigmoid(X_xor @ W1 + b1)  # hidden layer 출력
ax.scatter(H[:, 0], H[:, 1], c=colors, s=200, zorder=5, edgecolors='black')
for i in range(4):
    ax.annotate(f'XOR={y_xor[i]}', (H[i, 0], H[i, 1]), textcoords='offset points',
                xytext=(10, 10), fontsize=10)
# Decision boundary in hidden space
h_line = np.linspace(-0.1, 1.1, 100)
ax.plot(h_line, (10 + 20*h_line) / 20, 'k--', alpha=0.5, label='decision boundary')
ax.set_xlabel('h1 (NAND-like)'); ax.set_ylabel('h2 (OR-like)')
ax.set_title('Hidden Layer Space (linearly separable!)')
ax.set_xlim(-0.1, 1.1); ax.set_ylim(-0.1, 1.1)
ax.grid(True, alpha=0.3); ax.set_aspect('equal'); ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

# 최종 출력 확인
print("\nMLP XOR 결과:")
for i, x in enumerate(X_xor):
    h = sigmoid(x @ W1 + b1)
    out = sigmoid(h @ W2 + b2)
    print(f"  ({x[0]}, {x[1]}) -> hidden={np.round(h, 3)} -> output={out[0]:.4f} -> {int(out[0] > 0.5)}")

---
## 3. 활성화 함수 (Activation Functions)

활성화 함수는 신경망에 **비선형성**을 부여한다. 활성화 함수가 없으면 아무리 층을 쌓아도 결국 하나의 선형 변환과 같다.

| 함수 | 수식 | 특징 | 사용처 |
|------|------|------|--------|
| Sigmoid | $\sigma(x) = \frac{1}{1+e^{-x}}$ | 출력 (0,1), gradient vanishing | 이진 분류 출력 |
| Tanh | $\tanh(x) = \frac{e^x - e^{-x}}{e^x + e^{-x}}$ | 출력 (-1,1), zero-centered | RNN hidden |
| ReLU | $\max(0, x)$ | 간단, 빠름, dead neuron 문제 | CNN, MLP hidden |
| LeakyReLU | $\max(\alpha x, x)$ | dead neuron 해결 | ReLU 대안 |
| GELU | $x \cdot \Phi(x)$ | 부드러운 ReLU | Transformer (BERT, GPT) |

In [ ]:
x = np.linspace(-5, 5, 500)

# 활성화 함수 정의
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def tanh(x):
    return np.tanh(x)

def relu(x):
    return np.maximum(0, x)

def leaky_relu(x, alpha=0.1):
    return np.where(x > 0, x, alpha * x)

def gelu(x):
    return 0.5 * x * (1 + np.tanh(np.sqrt(2/np.pi) * (x + 0.044715 * x**3)))

# 도함수 정의
def sigmoid_deriv(x):
    s = sigmoid(x)
    return s * (1 - s)

def tanh_deriv(x):
    return 1 - np.tanh(x)**2

def relu_deriv(x):
    return np.where(x > 0, 1.0, 0.0)

def leaky_relu_deriv(x, alpha=0.1):
    return np.where(x > 0, 1.0, alpha)

activations = [
    ('Sigmoid', sigmoid, sigmoid_deriv, 'tab:blue'),
    ('Tanh', tanh, tanh_deriv, 'tab:orange'),
    ('ReLU', relu, relu_deriv, 'tab:green'),
    ('LeakyReLU (a=0.1)', leaky_relu, leaky_relu_deriv, 'tab:red'),
    ('GELU', gelu, None, 'tab:purple'),
]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 왼쪽: 활성화 함수
ax = axes[0]
for name, fn, _, color in activations:
    ax.plot(x, fn(x), label=name, color=color, linewidth=2)
ax.axhline(y=0, color='k', linewidth=0.5)
ax.axvline(x=0, color='k', linewidth=0.5)
ax.set_xlim(-5, 5); ax.set_ylim(-2, 5)
ax.set_xlabel('x'); ax.set_ylabel('f(x)')
ax.set_title('Activation Functions')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

# 오른쪽: 도함수 (GELU 제외)
ax = axes[1]
for name, _, deriv_fn, color in activations:
    if deriv_fn is not None:
        ax.plot(x, deriv_fn(x), label=f"{name}'", color=color, linewidth=2)
ax.axhline(y=0, color='k', linewidth=0.5)
ax.axvline(x=0, color='k', linewidth=0.5)
ax.set_xlim(-5, 5); ax.set_ylim(-0.5, 1.5)
ax.set_xlabel('x'); ax.set_ylabel("f'(x)")
ax.set_title('Derivatives (Gradient)')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("핵심 관찰:")
print("- Sigmoid/Tanh: 양쪽 끝에서 gradient -> 0 (vanishing gradient 문제)")
print("- ReLU: x > 0이면 gradient = 1 (vanishing 없음), x < 0이면 완전히 0 (dead neuron)")
print("- LeakyReLU: x < 0에서도 작은 gradient (dead neuron 해결)")
print("- GELU: 부드러운 ReLU, Transformer에서 표준으로 사용")

---
## 4. Universal Approximation Theorem

**정리**: 충분히 넓은(뉴런이 많은) 하나의 hidden layer를 가진 신경망은 어떤 연속 함수도 원하는 정밀도로 근사할 수 있다.

### 직관적 이해

1. Sigmoid 뉴런 하나는 **계단 함수**를 근사할 수 있다 (가중치를 크게 하면)
2. 두 개의 계단 함수를 조합하면 **범프(bump)** 함수를 만들 수 있다
3. 여러 bump 함수를 합치면 어떤 복잡한 함수도 근사할 수 있다

아래에서 뉴런 수를 늘려가며 sin(x) 함수를 얼마나 잘 근사하는지 본다.

In [ ]:
# Universal Approximation 시각화: 뉴런 수에 따른 함수 근사 능력
np.random.seed(42)

def train_simple_nn(n_hidden, n_steps=3000, lr=0.01):
    """1-hidden-layer NN으로 sin(x) 근사"""
    # 학습 데이터
    X_train = np.linspace(-2*np.pi, 2*np.pi, 100).reshape(-1, 1)
    y_train = np.sin(X_train)
    
    # 가중치 초기화
    W1 = np.random.randn(1, n_hidden) * 0.5
    b1 = np.zeros((1, n_hidden))
    W2 = np.random.randn(n_hidden, 1) * 0.5
    b2 = np.zeros((1, 1))
    
    for step in range(n_steps):
        # Forward
        z1 = X_train @ W1 + b1
        h = np.tanh(z1)
        y_pred = h @ W2 + b2
        
        # Loss (MSE)
        loss = np.mean((y_pred - y_train)**2)
        
        # Backward
        dL_dy = 2 * (y_pred - y_train) / len(X_train)
        dL_dW2 = h.T @ dL_dy
        dL_db2 = np.sum(dL_dy, axis=0, keepdims=True)
        dL_dh = dL_dy @ W2.T
        dL_dz1 = dL_dh * (1 - h**2)  # tanh derivative
        dL_dW1 = X_train.T @ dL_dz1
        dL_db1 = np.sum(dL_dz1, axis=0, keepdims=True)
        
        # Update
        W2 -= lr * dL_dW2
        b2 -= lr * dL_db2
        W1 -= lr * dL_dW1
        b1 -= lr * dL_db1
    
    return W1, b1, W2, b2, loss

fig, axes = plt.subplots(1, 4, figsize=(16, 3.5))
X_plot = np.linspace(-2*np.pi, 2*np.pi, 200).reshape(-1, 1)
y_true = np.sin(X_plot)

for ax, n_hidden in zip(axes, [2, 5, 10, 50]):
    W1, b1, W2, b2, loss = train_simple_nn(n_hidden, n_steps=5000, lr=0.01)
    h = np.tanh(X_plot @ W1 + b1)
    y_pred = h @ W2 + b2
    
    ax.plot(X_plot, y_true, 'b-', label='sin(x)', linewidth=2)
    ax.plot(X_plot, y_pred, 'r--', label='NN approx', linewidth=2)
    ax.set_title(f'{n_hidden} hidden neurons\nMSE={loss:.4f}')
    ax.set_xlim(-2*np.pi, 2*np.pi); ax.set_ylim(-1.5, 1.5)
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

plt.suptitle('Universal Approximation: More neurons = Better approximation', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

---
## 5. 2-Layer NN from Scratch (NumPy Only)

XOR 문제를 forward + backward를 직접 구현하여 풀어본다. PyTorch 없이 NumPy만 사용.

### 네트워크 구조
- 입력: 2개 (x1, x2)
- Hidden layer: 4개 뉴런, ReLU 활성화
- 출력: 1개, Sigmoid 활성화 (이진 분류)

### 수식

**Forward pass**:
$$z_1 = XW_1 + b_1, \quad h = \text{ReLU}(z_1)$$
$$z_2 = hW_2 + b_2, \quad \hat{y} = \sigma(z_2)$$

**Loss** (Binary Cross-Entropy):
$$L = -\frac{1}{N}\sum[y\log\hat{y} + (1-y)\log(1-\hat{y})]$$

**Backward pass** (chain rule):
$$\frac{\partial L}{\partial z_2} = \hat{y} - y$$
$$\frac{\partial L}{\partial W_2} = h^T \cdot \frac{\partial L}{\partial z_2}, \quad \frac{\partial L}{\partial b_2} = \sum \frac{\partial L}{\partial z_2}$$
$$\frac{\partial L}{\partial h} = \frac{\partial L}{\partial z_2} \cdot W_2^T$$
$$\frac{\partial L}{\partial z_1} = \frac{\partial L}{\partial h} \odot \mathbb{1}[z_1 > 0]$$
$$\frac{\partial L}{\partial W_1} = X^T \cdot \frac{\partial L}{\partial z_1}, \quad \frac{\partial L}{\partial b_1} = \sum \frac{\partial L}{\partial z_1}$$

In [ ]:
# ==========================================
# 2-Layer NN from scratch (NumPy only)
# ==========================================
np.random.seed(42)

# XOR 데이터
X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=float)
y = np.array([[0], [1], [1], [0]], dtype=float)

# 하이퍼파라미터
input_dim = 2
hidden_dim = 4
output_dim = 1
lr = 0.5
epochs = 5000

# 가중치 초기화 (He initialization)
W1 = np.random.randn(input_dim, hidden_dim) * np.sqrt(2.0 / input_dim)
b1 = np.zeros((1, hidden_dim))
W2 = np.random.randn(hidden_dim, output_dim) * np.sqrt(2.0 / hidden_dim)
b2 = np.zeros((1, output_dim))

def sigmoid(x):
    return 1 / (1 + np.exp(-np.clip(x, -500, 500)))

def relu(x):
    return np.maximum(0, x)

losses = []

for epoch in range(epochs):
    # ========== Forward Pass ==========
    z1 = X @ W1 + b1           # (4, 4)
    h = relu(z1)                # (4, 4)
    z2 = h @ W2 + b2           # (4, 1)
    y_hat = sigmoid(z2)        # (4, 1)
    
    # Loss (Binary Cross-Entropy)
    eps = 1e-8
    loss = -np.mean(y * np.log(y_hat + eps) + (1 - y) * np.log(1 - y_hat + eps))
    losses.append(loss)
    
    # ========== Backward Pass ==========
    N = X.shape[0]
    
    # dL/dz2 = y_hat - y (BCE + sigmoid 미분 결합)
    dz2 = (y_hat - y) / N                       # (4, 1)
    
    # dL/dW2 = h^T @ dz2
    dW2 = h.T @ dz2                              # (4, 1)
    db2 = np.sum(dz2, axis=0, keepdims=True)     # (1, 1)
    
    # dL/dh = dz2 @ W2^T
    dh = dz2 @ W2.T                              # (4, 4)
    
    # dL/dz1 = dh * relu_derivative
    dz1 = dh * (z1 > 0).astype(float)            # (4, 4)
    
    # dL/dW1 = X^T @ dz1
    dW1 = X.T @ dz1                              # (2, 4)
    db1 = np.sum(dz1, axis=0, keepdims=True)     # (1, 4)
    
    # ========== Update ==========
    W2 -= lr * dW2
    b2 -= lr * db2
    W1 -= lr * dW1
    b1 -= lr * db1
    
    if (epoch + 1) % 1000 == 0:
        print(f"Epoch {epoch+1:5d} | Loss: {loss:.6f}")

# 최종 결과
print("\n=== 학습 결과 ===")
for i in range(4):
    print(f"  Input: {X[i]} -> Predicted: {y_hat[i][0]:.4f} -> {int(y_hat[i][0] > 0.5)} (Target: {int(y[i][0])})")

In [ ]:
# Loss 곡선 시각화
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Loss curve
ax = axes[0]
ax.plot(losses)
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
ax.set_title('Training Loss (NumPy NN)')
ax.grid(True, alpha=0.3)

# Decision boundary
ax = axes[1]
xx, yy = np.meshgrid(np.linspace(-0.5, 1.5, 200), np.linspace(-0.5, 1.5, 200))
grid_input = np.c_[xx.ravel(), yy.ravel()]
z1_grid = grid_input @ W1 + b1
h_grid = relu(z1_grid)
z2_grid = h_grid @ W2 + b2
y_grid = sigmoid(z2_grid).reshape(xx.shape)

ax.contourf(xx, yy, y_grid, levels=50, cmap='RdBu_r', alpha=0.7)
ax.contour(xx, yy, y_grid, levels=[0.5], colors='black', linewidths=2)
colors_plot = ['red' if yi == 0 else 'blue' for yi in y.ravel()]
ax.scatter(X[:, 0], X[:, 1], c=colors_plot, s=200, edgecolors='black', zorder=5)
ax.set_xlabel('x1'); ax.set_ylabel('x2')
ax.set_title('Decision Boundary (XOR)')
ax.set_aspect('equal')

plt.tight_layout()
plt.show()

---
## 6. PyTorch nn.Module로 동일한 모델 구현

위에서 NumPy로 직접 구현한 것과 동일한 구조를 PyTorch로 구현하여 비교한다.

**PyTorch의 장점**: backward pass를 자동으로 계산해준다 (autograd).

In [ ]:
# PyTorch로 동일한 XOR 네트워크 구현
torch.manual_seed(42)

class XORNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Linear(2, 4)   # input -> hidden
        self.layer2 = nn.Linear(4, 1)   # hidden -> output
        self.relu = nn.ReLU()
        self.sigmoid = nn.Sigmoid()
    
    def forward(self, x):
        x = self.relu(self.layer1(x))    # hidden layer + ReLU
        x = self.sigmoid(self.layer2(x)) # output + Sigmoid
        return x

# 데이터 준비
X_torch = torch.tensor([[0,0],[0,1],[1,0],[1,1]], dtype=torch.float32)
y_torch = torch.tensor([[0],[1],[1],[0]], dtype=torch.float32)

# 모델, 손실함수, 옵티마이저
model = XORNet()
criterion = nn.BCELoss()  # Binary Cross-Entropy
optimizer = torch.optim.SGD(model.parameters(), lr=0.5)

# 학습
losses_pytorch = []
for epoch in range(5000):
    # Forward
    y_pred = model(X_torch)
    loss = criterion(y_pred, y_torch)
    losses_pytorch.append(loss.item())
    
    # Backward (PyTorch가 자동으로 gradient 계산!)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    if (epoch + 1) % 1000 == 0:
        print(f"Epoch {epoch+1:5d} | Loss: {loss.item():.6f}")

# 최종 결과
print("\n=== PyTorch 학습 결과 ===")
with torch.no_grad():
    predictions = model(X_torch)
    for i in range(4):
        pred = predictions[i].item()
        print(f"  Input: [{X_torch[i][0]:.0f}, {X_torch[i][1]:.0f}] -> Predicted: {pred:.4f} -> {int(pred > 0.5)} (Target: {int(y_torch[i].item())})")

In [ ]:
# NumPy vs PyTorch Loss 비교
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(losses, label='NumPy (from scratch)', alpha=0.7)
ax.plot(losses_pytorch, label='PyTorch (nn.Module)', alpha=0.7)
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
ax.set_title('NumPy vs PyTorch Training Comparison')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("비교:")
print("- NumPy: forward, backward, update를 모두 직접 구현 (교육 목적)")
print("- PyTorch: loss.backward()가 gradient를 자동 계산 (실전 사용)")
print("- 결과는 동일하지만 PyTorch가 훨씬 간결하고 GPU 활용 가능")

---
## 연습 문제

아래 문제를 직접 풀어보세요.

### 연습 1: 활성화 함수 변경 실험

위의 NumPy XOR 네트워크에서 ReLU 대신 Sigmoid를 hidden layer 활성화 함수로 사용하면 어떻게 되는지 실험하세요.

- Sigmoid의 도함수: $\sigma'(x) = \sigma(x)(1 - \sigma(x))$
- 수렴 속도가 어떻게 달라지는가?
- 최종 loss는 어떤가?

In [ ]:
# TODO: ReLU 대신 Sigmoid를 hidden layer에 사용한 XOR 네트워크를 구현하세요
# 힌트:
# 1. forward에서 relu(z1) -> sigmoid(z1) 변경
# 2. backward에서 relu 도함수 -> sigmoid 도함수 변경
#    dz1 = dh * sigmoid(z1) * (1 - sigmoid(z1))
# 3. loss curve를 ReLU 버전과 비교 플롯


### 연습 2: PyTorch로 Moon 데이터셋 분류

sklearn의 make_moons 데이터셋을 PyTorch MLP로 분류하세요.

In [ ]:
from sklearn.datasets import make_moons

X_moon, y_moon = make_moons(n_samples=500, noise=0.2, random_state=42)

# 데이터 확인
plt.scatter(X_moon[:, 0], X_moon[:, 1], c=y_moon, cmap='RdBu', s=10)
plt.title('Moon Dataset'); plt.grid(True, alpha=0.3)
plt.show()

# TODO: PyTorch nn.Module로 MLP 구현
# 1. 입력 2 -> hidden 16 (ReLU) -> hidden 8 (ReLU) -> 출력 1 (Sigmoid)
# 2. BCELoss + Adam optimizer (lr=0.01)
# 3. 1000 epoch 학습
# 4. Decision boundary 시각화


---
## 핵심 정리

| 개념 | 핵심 내용 |
|------|----------|
| Perceptron | 선형 분류기, AND/OR 가능, XOR 불가 |
| Multi-Layer Perceptron | Hidden layer로 비선형 결정 경계 생성, XOR 해결 |
| 활성화 함수 | 비선형성 부여. ReLU(일반), GELU(Transformer)가 주류 |
| Universal Approximation | 충분한 뉴런이 있으면 어떤 함수도 근사 가능 |
| Forward Pass | 입력 -> 가중치 곱 -> 활성화 -> 출력 |
| Backward Pass | Chain rule로 gradient 계산, 가중치 업데이트 |
| PyTorch nn.Module | backward를 자동 계산, 실전에서 사용 |

**다음 노트북**: [02-backpropagation.ipynb](02-backpropagation.ipynb) - 역전파와 autograd 구현